<a href="https://colab.research.google.com/github/Danny3636/Generative-AI-Tasks/blob/main/Required_Task_5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

**From this task and code, I've learned how to process text data by extracting and cleaning URLs, generate numerical representations (embeddings) of text using pre-trained models and built an interactive semantic search tool with Gradio to find similar articles based on these embeddings.**

# Task
Load the `financial_news.csv` dataset, extract URLs from the 'text' column into a new 'URL' column while cleaning the original 'text', generate sentence embeddings for the cleaned text, and then build a Gradio-based semantic search tool that uses these embeddings to find and display the top 5 most similar news articles (text and URL) for a given text query.

**Reasoning**:
The previous attempt to load the CSV file failed because the file path was incorrect. I will correct the file path to `/content/sample_data/financial_news.csv` to successfully load the dataset.



In [3]:
import pandas as pd

df = pd.read_csv('/content/sample_data/financial_news.csv')
df.head()

,text,label
0,Here are Thursday's biggest analyst calls: App...,0
1,Buy Las Vegas Sands as travel to Singapore bui...,0
2,"Piper Sandler downgrades DocuSign to sell, cit...",0
3,"Analysts react to Tesla's latest earnings, bre...",0
4,Netflix and its peers are set for a ‘return to...,0


**Reasoning**:
Now that the dataset is loaded, I will extract URLs from the 'text' column using a regular expression, store them in a new 'URL' column, and then remove these URLs from the original 'text' column to clean it. Finally, I'll display the first few rows of the updated DataFrame to verify the changes.



In [4]:
import re

# Define a function to extract URLs and clean text
def extract_and_clean_urls(text):
    url_pattern = r'https?://[a-zA-Z0-9./-]+'
    urls = re.findall(url_pattern, text)
    cleaned_text = re.sub(url_pattern, '', text)
    return cleaned_text, ';'.join(urls)

# Apply the function to the 'text' column
df[['text', 'URL']] = df['text'].apply(lambda x: pd.Series(extract_and_clean_urls(x)))

# Display the first few rows of the DataFrame with the new 'URL' column and cleaned 'text'
df.head()

,text,label,URL
0,Here are Thursday's biggest analyst calls: App...,0,https://t.co/QPN8Gwl7Uh
1,Buy Las Vegas Sands as travel to Singapore bui...,0,https://t.co/fLS2w57iCz
2,"Piper Sandler downgrades DocuSign to sell, cit...",0,https://t.co/1EmtywmYpr
3,"Analysts react to Tesla's latest earnings, bre...",0,https://t.co/kwhoE6W06u
4,Netflix and its peers are set for a ‘return to...,0,https://t.co/jPpdl0D9s4


In [5]:
pip install sentence_transformers

In [6]:
from sentence_transformers import SentenceTransformer

# Load a pre-trained sentence transformer model
model = SentenceTransformer('all-MiniLM-L6-v2')

# Generate embeddings for the cleaned text
df['embeddings'] = df['text'].apply(lambda x: model.encode(x))

# Display the shape of the embeddings to confirm
print(f"Shape of embeddings: {df['embeddings'].iloc[0].shape}")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Shape of embeddings: (384,)


## Build Semantic Search Tool with Gradio

### Subtask:
Develop a Gradio interface for a semantic search tool. The tool will allow users to input text (e.g., 'earnings surprise', 'regulatory fine'). The tool will then calculate the cosine similarity between the input text's embedding and the pre-computed sentence embeddings, outputting the top 5 closest records (including original text and URL).


In [7]:
import gradio as gr
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

# Convert the embeddings Series to a NumPy array for efficient calculation
corpus_embeddings = np.array(df['embeddings'].tolist())

def semantic_search(query):
    # Encode the input query
    query_embedding = model.encode(query)

    # Calculate cosine similarity between query and all corpus embeddings
    similarities = cosine_similarity([query_embedding], corpus_embeddings)[0]

    # Get the indices of the top 5 most similar articles
    top_indices = similarities.argsort()[-5:][::-1]

    results_str = ""
    for i, idx in enumerate(top_indices):
        article_text = df.iloc[idx]['text']
        article_url = df.iloc[idx]['URL']
        similarity_score = similarities[idx]
        results_str += f"**Rank {i+1} (Similarity: {similarity_score:.4f}):**\n"
        results_str += f"Text: {article_text}\n"
        results_str += f"URL: {article_url}\n\n"
    return results_str

# Create the Gradio interface
iface = gr.Interface(
    fn=semantic_search,
    inputs=gr.Textbox(lines=2, placeholder='Enter your search query here...'),
    outputs=gr.Markdown(),
    title='Semantic Search for Financial News',
    description='Enter a query to find the most semantically similar financial news articles.'
)

# Launch the interface
iface.launch()

It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://16ec4cad316028b410.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


## Final Task

### Subtask:
Summarize the steps taken to process the data and create the semantic search tool, providing instructions on how to use the Gradio application.


## Summary:

### Data Analysis Key Findings

*   The `financial_news.csv` dataset was successfully loaded and processed.
*   URLs were extracted from the 'text' column into a new 'URL' column, and the original 'text' column was cleaned by removing these URLs.
*   Sentence embeddings were generated for the cleaned text using the 'all-MiniLM-L6-v2' pre-trained model from the `sentence_transformers` library. Each article's text now has a numerical representation in the 'embeddings' column.
*   A Gradio-based semantic search tool was developed, allowing users to query financial news articles.
*   The tool uses cosine similarity to compare the user's query embedding with the generated article embeddings and retrieves the top 5 most similar articles, displaying their cleaned text and extracted URLs.

### Insights or Next Steps

*   **Gradio Application Usage**: To use the semantic search tool, users should enter a text query into the provided textbox within the Gradio interface. The tool will then display the top 5 most semantically similar financial news articles, including their text and associated URLs, along with their similarity scores.
*   **Scalability & Performance**: For larger datasets, consider optimizing embedding storage (e.g., using a vector database) and similarity search algorithms to improve performance.
